# Rule of Law

This notebook loads the rule of law index data, inverts it and creates ISO3 codes columns using ISO2. Subsequently, the notebook merges the data with country boundaries, rasterizes it using the bii layer as reference grid and creates: 
- a raster (with bii as reference raster)
- a map figure (`OUT_PNG`)
  
## How to run
1. Put the required input files in the same folder as this notebook (or edit the paths in the **Configuration** cell below).
2. Run the cells from top to bottom.

## Required files
- `rule_of_law.xlsx`
- `bii_5000m.tif`
- `World_Countries_(Generalized)_8414823838130214587.gpkg` (or your country layer)

In [ ]:
# Configuration (edit these paths if needed)

RULE_OF_LAW = "rule_of_law.xlsx"  
WORLD_COUNTRIES_GENERAL = "World_Countries_(Generalized)_8414823838130214587.gpkg" 
BII_5000M = 'sensitivity\\biodiversity_intactness\\bii_5000m.tif'  # TODO: update path
RULE_OF_LAW_5000M = r"rule_of_law_5000m.tif" 
RULE_OF_LAW = "rule_of_law.png" 


In [ ]:
#import packages
import pandas as pd
import geopandas as gpd
import pycountry
import rasterio
from rasterio.transform import from_origin
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from matplotlib.patches import Patch
from rasterio.windows import Window
from rasterio.features import rasterize

In [ ]:
# Read excel with no header
df = pd.read_excel(RULE_OF_LAW, header=None)

# use first column as row index 
df = df.set_index(0)

# Transpose
df_long = df.T

# rename column names
df_long = df_long.rename(columns={
    "Country": "Country",
    "Country Code": "Country_Code",
    "Region": "Region",
    "Income Group*": "Income_Group",
    "WJP Rule of Law Index: Overall Score": "RoL_score"
})


In [ ]:
#invert rule of law index
df_long['RoL_score_inverted']=1-df_long['RoL_score']

In [ ]:
# Function to convert iso3 in iso2
def iso3_to_iso2(iso3):
    if iso3 == "XKX":      # Kosovo special case
        return "XK"

    try:
        country = pycountry.countries.get(alpha_3=iso3)
        if country is not None:
            return country.alpha_2
    except:
        pass

    return None

# Apply function to dataframe
df_long['ISO2'] = df_long['Country_Code'].apply(iso3_to_iso2)

#print missing matches to check
missing = df_long[df_long['ISO2'].isna()]
print(missing)


In [ ]:
# Assign geometry to rule of law

# Load GeoPackage
gdf = gpd.read_file(WORLD_COUNTRIES_GENERAL)

# Ensure geometry column is set
gdf = gdf.set_geometry('geometry')

# Merge rule of law with country boundaries
gdf = gdf.merge(
    df_long[['ISO2', 'RoL_score_inverted']],
    left_on='ISO',
    right_on='ISO2',
    how='left'
)

# keep only relevant columns
gdf = gdf[['COUNTRY', 'ISO', 'RoL_score_inverted', 'geometry']]

# Check result
print(gdf.head())


In [ ]:
# rasterize poverty variable using bii as reference raster

#paths
ref = BII_5000M #reference raster
out_path = RULE_OF_LAW_5000M

# Open reference raster as the template grid
with rasterio.open(ref) as src:
    profile = src.profile.copy()
    crs = src.crs
    transform = src.transform
    out_shape = (src.height, src.width)

    # reproject polygons to match raster CRS
    gdf_r = gdf.to_crs(crs)


    # Write output windowed processing
    with rasterio.open(out_path, "w", **profile) as dst:
        for block_index, window in src.block_windows(1):    
            win_transform = rasterio.windows.transform(window, transform)
        
            window_raster = rasterize(
                    [(geom, value) for geom, value in zip(gdf_r.geometry, gdf_r.RoL_score_inverted)],
                    out_shape=(window.height, window.width),  
                    transform=win_transform,
                    fill=-9999,
                    dtype="float32"
                )
    
            dst.write(window_raster, 1, window=window)

print("Saved:", out_path)

In [ ]:
# plot

# paths
raster_path = RULE_OF_LAW_5000M
countries_path = WORLD_COUNTRIES_GENERAL
out_png = RULE_OF_LAW

max_width = 5000

# load raster 
with rasterio.open(raster_path) as src:
    bounds = src.bounds
    crs = src.crs
    nodata_val = src.nodata
   
    #calculate display size
    scale = max_width / src.width
    out_w = int(src.width * scale)
    out_h = int(src.height * scale)
    #read raster band with new size using .nearest resampling
    arr = src.read(
        1,
        out_shape=(out_h, out_w),
        resampling=Resampling.nearest
    ).astype("float32")
    # update transform after resampling
    transform = src.transform * src.transform.scale(
        src.width / out_w,
        src.height / out_h
    )
#open country boundaries gpkg
world = gpd.read_file(countries_path).to_crs(crs)
world = world[world["COUNTRY"] != "Antarctica"]

# build country mask
country_mask = geometry_mask(
    geometries=world.geometry,
    transform=transform,
    invert=True,          
    out_shape=arr.shape
)


#define nodata
nodata = (arr == -9999)

#convert -9999 to nan for plotting
arr = arr.astype("float32")
arr[arr == -9999] = np.nan

#mask data outside country and no data for arr
masked = np.ma.masked_where((~country_mask) | nodata, arr)
#transparency settings
alpha = np.where(country_mask, 0.95, 0.0)

#define valid values
valid = arr[country_mask & np.isfinite(arr)]

# robust min/max from valid pixels
vmin = float(valid.min())
vmax = float(valid.max())


# colormaps
nodata_color = "#D1D5DB"
cmap = LinearSegmentedColormap.from_list(
    "poverty_red_grad",
    ["#FFF1F2", "#FECACA","#F87171","#DC2626","#7F1D1D"])
cmap.set_bad(color=nodata_color)

#build figure, set size and background color
fig, ax = plt.subplots(figsize=(14, 7), dpi=200)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

#plot country boundaries, set color and linewidth
world.plot(
    ax=ax,
    facecolor="#F6F7F9",
    edgecolor="#B9C0C8",
    linewidth=0.35,
    zorder=1
)

#plot raster
raster = ax.imshow(
    masked,
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
    interpolation="nearest",
    alpha=alpha,#transparency
    zorder=2 #plot on top of country boundaries
)

# Plot country boundaries on top
world.boundary.plot(ax=ax, color="#695C5A", linewidth=0.3, zorder=2)

# title
ax.set_title("Rule of law", fontsize=18, fontweight="semibold", pad=14)
#no axis
ax.set_axis_off()

# colorbar 
cbar = plt.colorbar(raster, ax=ax, fraction=0.03, pad=0.02)#width of color bar relative to plot and space between plot and colorbar
cbar.set_label("Rule of law index (inverted=", fontsize=16, color="#2B2F36")
cbar.ax.tick_params(labelsize=10, colors="#2B2F36")#numbers on colorbar
cbar.outline.set_edgecolor("#E3E6EA")#edgecolor of colorbar
cbar.outline.set_linewidth(1.0)
cbar.ax.set_facecolor("white")

# NoData legend patch (grey)
legend_handles = [Patch(facecolor=nodata_color, edgecolor="none", label="No data")]
leg = ax.legend(
    handles=legend_handles,
    loc="lower left",
    frameon=True,
    framealpha=1,
    facecolor="white",
    edgecolor="#E3E6EA",
    borderpad=0.8,
    handlelength=1.2,
)

plt.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
